# Phase 4: open-VLM reasoner -> plans -> pixels (Colab)

The full loop with **no API keys**: Qwen2.5-VL reads each grayscale image + context prompt and selects objects/modifiers; the KB resolves colours; Grounded-SAM grounds the plan; the naive renderer + adherence evaluator score it.

GPU: A100/L4 recommended (7B model). On T4, switch to `Qwen/Qwen2.5-VL-3B-Instruct`.

> Runtime → GPU before running.

In [ ]:
!nvidia-smi -L
%cd /content
!test -d chroma-reasoner || git clone https://github.com/tomqi6195/chroma-reasoner.git
!pip install -q -e chroma-reasoner
!pip install -q "transformers<5" accelerate pycocotools
import sys; sys.path.insert(0, '/content/chroma-reasoner/src')
import transformers; print('transformers', transformers.__version__)

## 1. Images + grayscale (same 5 as Phase 2)

In [ ]:
import glob, json, os, urllib.request
import cv2

IMAGE_IDS = ['000000000139', '000000001000', '000000002299', '000000010092', '000000022755']
# abstract context prompts - the reasoner's actual input
CONTEXT = {
    '000000000139': '',
    '000000001000': 'summer tennis camp group photo',
    '000000002299': 'British school class photo, late 1940s, overcast day',
    '000000010092': 'tropical jungle lodge interior',
    '000000022755': 'overcast day in an American town',
}

os.makedirs('images', exist_ok=True); os.makedirs('gray', exist_ok=True)
for iid in IMAGE_IDS:
    dest = f'images/{iid}.jpg'
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f'http://images.cocodataset.org/val2017/{iid}.jpg', dest)
    img = cv2.imread(dest)
    cv2.imwrite(f'gray/{iid}.png', cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[:, :, 0])
print('ready')

## 2. Reason plans with Qwen2.5-VL (no API key)

In [ ]:
from chroma_reasoner.kb import load_kb
from chroma_reasoner.reasoner import reason_plan
from chroma_reasoner.reasoner.backend_open import QwenVLBackend
from chroma_reasoner.reasoner.planner import ReasonerError

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'   # T4: 'Qwen/Qwen2.5-VL-3B-Instruct'
kb = load_kb('/content/chroma-reasoner/kb')
backend = QwenVLBackend(model_id=MODEL_ID)   # ~17 GB VRAM for 7B bf16
print('backend ready')

In [ ]:
from chroma_reasoner.plan import LabColor, lab_to_hex

os.makedirs('plans/reasoned', exist_ok=True)
for iid in IMAGE_IDS:
    try:
        plan = reason_plan(kb, backend, f'gray/{iid}.png', user_prompt=CONTEXT[iid])
    except ReasonerError as e:
        print(f'!! {iid}: selection unrepairable:'); print(e); continue
    with open(f'plans/reasoned/{iid}.json', 'w') as f:
        json.dump(plan, f, indent=2)
    print(f'== {iid}  prompt={CONTEXT[iid]!r}')
    for r in plan['regions']:
        lab = LabColor.from_plan(r['resolved_colour'])
        mods = ', '.join(f"{m['family']}:{m['value']}" for m in r.get('modifiers', []))
        print(f"   [{r['object']:>16}] {lab_to_hex(lab)} conf={r['confidence']:.2f} [{mods}]  {r['grounding_phrase']}")

In [ ]:
# Free the VLM before loading Grounded-SAM (VRAM headroom on smaller GPUs)
import gc, torch
del backend
gc.collect(); torch.cuda.empty_cache()
print(f'free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 3. Ground the reasoned plans (Grounding DINO + SAM)

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import (AutoModelForZeroShotObjectDetection, AutoProcessor,
                          SamModel, SamProcessor)

device = 'cuda'
dino_proc = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-base')
dino = AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-base').to(device).eval()
sam_proc = SamProcessor.from_pretrained('facebook/sam-vit-huge')
sam = SamModel.from_pretrained('facebook/sam-vit-huge').to(device).eval()

@torch.no_grad()
def phrase_to_mask(image_pil, phrase):
    text = phrase.lower().rstrip('.') + '.'
    inputs = dino_proc(images=image_pil, text=text, return_tensors='pt').to(device)
    out = dino(**inputs)
    res = dino_proc.post_process_grounded_object_detection(
        out, inputs.input_ids, threshold=0.25, text_threshold=0.2,
        target_sizes=[image_pil.size[::-1]])[0]
    if len(res['boxes']) == 0:
        return None, 0.0
    best = res['scores'].argmax()
    box = res['boxes'][best].tolist()
    s_in = sam_proc(image_pil, input_boxes=[[box]], return_tensors='pt').to(device)
    s_out = sam(**s_in)
    masks = sam_proc.image_processor.post_process_masks(
        s_out.pred_masks.cpu(), s_in['original_sizes'].cpu(), s_in['reshaped_input_sizes'].cpu())[0][0]
    scores = s_out.iou_scores.cpu()[0, 0]
    return masks[scores.argmax()].numpy().astype(bool), float(res['scores'][best])

In [ ]:
from chroma_reasoner.plan import load_plan
from chroma_reasoner.plan.masks import region_key, save_mask

for plan_path in sorted(glob.glob('plans/reasoned/*.json')):
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    pil = Image.fromarray(np.stack([gray] * 3, axis=-1))
    for region in plan['regions']:
        mask, score = phrase_to_mask(pil, region['grounding_phrase'])
        if mask is None:
            print(f"!! {iid}/{region_key(region)}: NO DETECTION"); continue
        save_mask(mask, 'masks_reasoned', iid, region)
        print(f'{iid}/{region_key(region)}: dino {score:.2f}, {mask.sum()} px')

In [ ]:
# Re-resolve every colour at the MASK-MEASURED luminance (deterministic; no
# model). Open-VLM L estimates err by up to dL 60, and identical measured L
# across supposedly-different regions exposes grounding failures.
from chroma_reasoner.plan.masks import load_masks
from chroma_reasoner.reasoner import re_resolve_with_masks

for plan_path in sorted(glob.glob('plans/reasoned/*.json')):
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    try:
        masks = load_masks('masks_reasoned', iid, plan, shape=gray.shape)
    except FileNotFoundError as e:
        print(f'{iid}: {e}'); continue
    plan = re_resolve_with_masks(kb, plan, gray, masks)
    with open(plan_path, 'w') as f:
        json.dump(plan, f, indent=2)
    ls = {r['id']: r['resolved_colour']['L'] for r in plan['regions']}
    print(iid, 'measured L per region:', ls)
    # suspicious: several regions with near-identical measured L usually means
    # their grounding phrases grabbed the same (wrong) box - tighten phrases.

## 4. Render + adherence (text -> image, end to end)

In [ ]:
import matplotlib.pyplot as plt
from chroma_reasoner.plan.adherence import evaluate_adherence
from chroma_reasoner.plan.hints import render_naive
from chroma_reasoner.plan.masks import load_masks

os.makedirs('results/reasoned_naive', exist_ok=True)
for plan_path in sorted(glob.glob('plans/reasoned/*.json')):
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    try:
        masks = load_masks('masks_reasoned', iid, plan, shape=gray.shape)
    except FileNotFoundError as e:
        print(f'{iid}: {e}'); continue
    rgb = render_naive(gray, masks, plan)
    cv2.imwrite(f'results/reasoned_naive/{iid}.png', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    rep = evaluate_adherence(rgb, masks, plan)
    print(iid, '| mean dE', rep['mean_delta_e'], '| pass', rep['n_pass'], '/', rep['n_regions'])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(cv2.imread(f'images/{iid}.jpg'), cv2.COLOR_BGR2RGB)); axes[0].set_title('original'); axes[0].axis('off')
    axes[1].imshow(rgb); axes[1].set_title(f'reasoned: {CONTEXT[iid] or "(no prompt)"}'); axes[1].axis('off')
    plt.show()

## 5. Export

In [ ]:
!zip -rq phase4_outputs.zip plans/reasoned masks_reasoned results/reasoned_naive
from google.colab import files
files.download('phase4_outputs.zip')
# Optional next step: feed plans + masks into the Phase-2 Control Color cells
# (notebooks/phase2_masks_colab.ipynb section 4) for the diffusion render.